In [1]:
import GEOparse, pandas as pd

gse = GEOparse.get_GEO(geo="GSE100927", destdir="../data/bronze")

23-Nov-2026 12:58:06 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE100nnn/GSE100927/soft/GSE100927_family.soft.gz to ../data/bronze\GSE100927_family.soft.gz
100%|██████████| 35.4M/35.4M [00:17<00:00, 2.14MB/s]
23-Nov-2026 12:58:25 DEBUG downloader - Size validation passed
23-Nov-2026 12:58:25 DEBUG downloader - Moving C:\Users\WINDOW~1\AppData\Local\Temp\tmpyxw8j54u to C:\Users\Windows11\Documents\data\bronze\GSE100927_family.soft.gz
23-Nov-2026 12:58:25 DEBUG downloader - Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE100nnn/GSE100927/soft/GSE100927_family.soft.gz
23-Nov-2026 12:58:25 INFO GEOparse - Parsing ../data/bronze\GSE100927_family.soft.gz: 
23-Nov-2026 12:58:25 DEBUG GEOparse - DATABASE: GeoMiame
23-Nov-2026 12:58:25 DEBUG GEOparse - SERIES: GSE100927
23-Nov-2026 12:58:25 DEBUG GEOparse - PLATFORM: GPL17077
C:\Users\Windows11\miniforge3\envs\perio-athero\Lib\site-packages\GEOparse\GEOparse.py:401: DtypeWarning: Columns (2: CONTROL_TYPE

In [2]:
print("Plateformes (GPL):", list(gse.gpls.keys()))
expr = gse.pivot_samples("VALUE")
pheno = gse.phenotype_data
print("expr shape:", expr.shape)
print("pheno shape:", pheno.shape)
print(pheno.columns.tolist())

Plateformes (GPL): ['GPL17077']
expr shape: (47264, 104)
pheno shape: (104, 31)
['title', 'geo_accession', 'status', 'submission_date', 'last_update_date', 'type', 'channel_count', 'source_name_ch1', 'organism_ch1', 'taxid_ch1', 'characteristics_ch1.0.tissue', 'characteristics_ch1.1.gender', 'molecule_ch1', 'extract_protocol_ch1', 'label_ch1', 'label_protocol_ch1', 'hyb_protocol', 'scan_protocol', 'description', 'data_processing', 'platform_id', 'contact_name', 'contact_email', 'contact_institute', 'contact_address', 'contact_city', 'contact_zip/postal_code', 'contact_country', 'supplementary_file', 'series_id', 'data_row_count']


In [3]:
pheno.iloc[0]

title                                                     H1_87: Femoral artery_3
geo_accession                                                          GSM2696609
status                                                      Public on Mar 07 2018
submission_date                                                       Jul 07 2017
last_update_date                                                      Mar 07 2018
type                                                                          RNA
channel_count                                                                   1
source_name_ch1                                    Atherosclerotic femoral artery
organism_ch1                                                         Homo sapiens
taxid_ch1                                                                    9606
characteristics_ch1.0.tissue                       Atherosclerotic femoral artery
characteristics_ch1.1.gender                                                 male
molecule_ch1    

In [4]:
print(pheno['characteristics_ch1.0.tissue'].value_counts())
print(pheno['characteristics_ch1.1.gender'].value_counts())

characteristics_ch1.0.tissue
Atherosclerotic carotid artery            29
Atherosclerotic femoral artery            26
Atherosclerotic infra-popliteal artery    14
Control femoral artery                    12
Control carotid artery                    12
Control infra-popliteal artery            11
Name: count, dtype: int64
characteristics_ch1.1.gender
male      85
female    19
Name: count, dtype: int64


In [5]:
print("NaN dans expr:", int(expr.isna().sum().sum()))
print(expr.iloc[:4, :3])
print(expr.stack().describe())

NaN dans expr: 0
name            GSM2696609  GSM2696610  GSM2696611
ID_REF                                            
A_19_P00315452        4.45        4.47        5.16
A_19_P00315459        4.99        5.12        4.56
A_19_P00315482        4.00        4.06        3.92
A_19_P00315492        4.13        4.41        4.15
count    4.915456e+06
mean     4.992957e+00
std      1.539249e+00
min      2.280000e+00
25%      3.850000e+00
50%      4.320000e+00
75%      5.780000e+00
max      1.410000e+01
dtype: float64


In [6]:
gpl = gse.gpls['GPL17077'].table
print(gpl.shape)
print(gpl.columns.tolist())
print(gpl.head(3))

(50739, 16)
['ID', 'SPOT_ID', 'CONTROL_TYPE', 'REFSEQ', 'GB_ACC', 'LOCUSLINK_ID', 'GENE_SYMBOL', 'GENE_NAME', 'UNIGENE_ID', 'ENSEMBL_ID', 'ACCESSION_STRING', 'CHROMOSOMAL_LOCATION', 'CYTOBAND', 'DESCRIPTION', 'GO_ID', 'SEQUENCE']
                ID       SPOT_ID CONTROL_TYPE     REFSEQ     GB_ACC  \
0  GE_BrightCorner       CONTROL          pos        NaN        NaN   
1       DarkCorner       CONTROL          pos        NaN        NaN   
2     A_23_P117082  A_23_P117082        FALSE  NM_015987  NM_015987   

   LOCUSLINK_ID GENE_SYMBOL               GENE_NAME UNIGENE_ID  \
0           NaN         NaN                     NaN        NaN   
1           NaN         NaN                     NaN        NaN   
2       50865.0       HEBP1  heme binding protein 1  Hs.642618   

        ENSEMBL_ID                                   ACCESSION_STRING  \
0              NaN                                                NaN   
1              NaN                                                NaN   
2

In [7]:
import numpy as np
from pathlib import Path

tissue = pheno['characteristics_ch1.0.tissue']
pheno_clean = pd.DataFrame({
    'sample_id': pheno['geo_accession'],
    'group': np.where(tissue.str.startswith('Atherosclerotic'), 'disease', 'control'),
    'site': tissue.str.replace('Atherosclerotic ', '', regex=False)
                  .str.replace('Control ', '', regex=False),
    'sex': pheno['characteristics_ch1.1.gender'],
    'age': np.nan,
    'platform': pheno['platform_id'],
}).reset_index(drop=True)

print(pheno_clean['group'].value_counts())
print(pd.crosstab(pheno_clean['group'], pheno_clean['site']))
assert set(expr.columns) == set(pheno_clean['sample_id'])
expr = expr[pheno_clean['sample_id']]      # même ordre que pheno

group
disease    69
control    35
Name: count, dtype: int64
site     carotid artery  femoral artery  infra-popliteal artery
group                                                          
control              12              12                      11
disease              29              26                      14


In [8]:
ann = gpl[(gpl['CONTROL_TYPE'] == 'FALSE') & gpl['GENE_SYMBOL'].notna()]
ann = ann[ann['GENE_SYMBOL'].str.strip() != ''].drop_duplicates('ID')
mapping = ann.set_index('ID')['GENE_SYMBOL']

e = expr.loc[expr.index.intersection(mapping.index)].copy()
e['gene'] = e.index.map(mapping)
expr_gene = e.groupby('gene').median()
expr_gene.columns.name = None

print("sondes avant:", expr.shape[0], "| sondes annotées:", e.shape[0],
      "| gènes uniques:", expr_gene.shape[0])
print(expr_gene.loc[expr_gene.index.isin(['BTK', 'ITGAL'])].iloc[:, :3])

sondes avant: 47264 | sondes annotées: 27833 | gènes uniques: 21863
       GSM2696609  GSM2696610  GSM2696611
gene                                     
BTK          5.92        6.77        5.22
ITGAL        5.08        5.55        4.80


In [9]:
out = Path('../data/silver/GSE100927')
out.mkdir(parents=True, exist_ok=True)
expr_gene.reset_index().to_parquet(out / 'expr.parquet')    # colonne 'gene' + 104 échantillons
pheno_clean.to_parquet(out / 'pheno.parquet')

chk = pd.read_parquet(out / 'expr.parquet')
print(chk.shape, chk.columns[:3].tolist())

(21863, 105) ['gene', 'GSM2696609', 'GSM2696610']


In [10]:
print(pd.read_parquet(out / 'pheno.parquet').head())

    sample_id    group                    site   sex  age  platform
0  GSM2696609  disease          femoral artery  male  NaN  GPL17077
1  GSM2696610  disease          femoral artery  male  NaN  GPL17077
2  GSM2696611  control          femoral artery  male  NaN  GPL17077
3  GSM2696612  control          carotid artery  male  NaN  GPL17077
4  GSM2696613  control  infra-popliteal artery  male  NaN  GPL17077
